In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_10.csv"
)

FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]


def create_masked_dataset(df, features):

    data = df.copy()

    mask_columns = []

    for feature in features:

        mask_name = f"{feature}_mask"

        # 1 = observed, 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_columns.append(mask_name)

    # Replace missing numerical values with neutral placeholder
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_columns


masked_df, MASK_FEATURES = create_masked_dataset(
    df,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Original input features:", len(FEATURES))
print("Mask features:", len(MASK_FEATURES))
print("Total model features:", len(MODEL_FEATURES))

print(
    "NaNs remaining:",
    masked_df[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

Original input features: 17
Mask features: 17
Total model features: 34
NaNs remaining: 0


In [2]:
masked_df[
        [
            "Sensor_2",
            "Sensor_2_mask",
            "Sensor_7",
            "Sensor_7_mask",
            "Sensor_20",
            "Sensor_20_mask"
        ]
    ].head(15)

,Sensor_2,Sensor_2_mask,Sensor_7,Sensor_7_mask,Sensor_20,Sensor_20_mask
0,0.183735,1.0,0.726248,1.0,0.713178,1.0
1,0.283133,1.0,0.628019,1.0,0.666667,1.0
2,0.343374,1.0,0.710145,1.0,0.627907,1.0
3,0.343374,1.0,0.740741,1.0,0.573643,1.0
4,0.349398,1.0,0.000000,0.0,0.589147,1.0
5,0.268072,1.0,0.776167,1.0,0.651163,1.0
6,0.382530,1.0,0.000000,0.0,0.744186,1.0
7,0.406626,1.0,0.644122,1.0,0.643411,1.0
8,0.274096,1.0,0.618357,1.0,0.705426,1.0
9,0.150602,1.0,0.602254,1.0,0.627907,1.0


Implementation

In [3]:
import pandas as pd
import numpy as np

train_missing = pd.read_csv(
    "Dataset/Missing_Train/processed_train_missing_10.csv"
)

test_missing = pd.read_csv(
    "Dataset/Missing_Test/processed_test_missing_10.csv"
)

In [4]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"
WINDOW = 30

In [5]:
def create_masked_dataset(df, features):

    data = df.copy()

    mask_features = []

    for feature in features:

        mask_name = feature + "_mask"

        # 1 = observed
        # 0 = missing
        data[mask_name] = (
            data[feature]
            .notna()
            .astype("float32")
        )

        mask_features.append(mask_name)

    # Placeholder only.
    # Mask tells model whether 0 is real or missing.
    data[features] = (
        data[features]
        .fillna(0.0)
        .astype("float32")
    )

    return data, mask_features

In [6]:
train_masked, MASK_FEATURES = create_masked_dataset(
    train_missing,
    FEATURES
)

test_masked, _ = create_masked_dataset(
    test_missing,
    FEATURES
)

MODEL_FEATURES = FEATURES + MASK_FEATURES

print("Input features:", len(MODEL_FEATURES))
# 34

Input features: 34


In [7]:
print(
    train_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

print(
    test_masked[MODEL_FEATURES]
    .isna()
    .sum()
    .sum()
)

0
0


In [8]:
clean_train = pd.read_csv(
    "Dataset/processed_train.csv"
)

np.random.seed(42)

engine_ids = clean_train[
    "Engine_ID"
].unique().copy()

np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]

In [9]:
train_df_30 = train_masked[
    train_masked["Engine_ID"].isin(train_ids)
].copy()

val_df_30 = train_masked[
    train_masked["Engine_ID"].isin(val_ids)
].copy()

In [10]:
def create_sequences(
    data,
    features,
    target,
    window=30
):

    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        targets = (
            engine_data[target]
            .values
            .astype("float32")
        )

        for i in range(window, len(engine_data)):

            X.append(
                values[i-window:i]
            )

            y.append(
                targets[i]
            )

    return np.array(X), np.array(y)

In [11]:
X_train_30, y_train_30 = create_sequences(
    train_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

X_val_30, y_val_30 = create_sequences(
    val_df_30,
    MODEL_FEATURES,
    TARGET,
    WINDOW
)

print(X_train_30.shape)
print(X_val_30.shape)

(12216, 30, 34)
(2720, 30, 34)


BiLSTM

In [12]:
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dropout,
    Dense
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

tf.keras.backend.clear_session()
tf.random.set_seed(42)

bilstm_30 = Sequential([

    Input(
        shape=(
            WINDOW,
            len(MODEL_FEATURES)
        )
    ),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [13]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history = bilstm_30.fit(
    X_train_30,
    y_train_30,

    validation_data=(
        X_val_30,
        y_val_30
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 17s 43ms/step - loss: 4788.2866 - mae: 58.7060 - val_loss: 2158.6470 - val_mae: 41.3359 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - loss: 1829.7994 - mae: 38.0649 - val_loss: 1748.9099 - val_mae: 37.2118 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - loss: 1770.5118 - mae: 37.2796 - val_loss: 1748.5862 - val_mae: 37.1645 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 35ms/step - loss: 1764.9656 - mae: 37.2460 - val_loss: 1748.5938 - val_mae: 37.1731 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - loss: 1769.4608 - mae: 37.2504 - val_loss: 1748.6520 - val_mae: 37.1883 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - loss: 1765.0720 - mae: 37.2045 - val_loss: 1748.5348 - val_mae: 37.1700 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - loss: 1771.5985 - mae: 37.2410 

In [14]:
def create_final_test_sequences(
    test_df,
    features,
    window=30
):

    X = []
    engine_ids = []

    for engine_id, engine_data in test_df.groupby(
        "Engine_ID"
    ):

        engine_data = engine_data.sort_values(
            "Cycle"
        )

        values = (
            engine_data[features]
            .values
            .astype("float32")
        )

        if len(values) >= window:

            X.append(
                values[-window:]
            )

            engine_ids.append(
                engine_id
            )

    return (
        np.array(X),
        np.array(engine_ids)
    )

In [15]:
X_test_30, test_engine_ids = (
    create_final_test_sequences(
        test_masked,
        MODEL_FEATURES,
        WINDOW
    )
)

print(X_test_30.shape)

(100, 30, 34)


In [16]:
bilstm_pred_30 = bilstm_30.predict(
    X_test_30,
    verbose=0
).ravel()

In [17]:
RUL_CAP = 125

rul_df = pd.read_csv(
    "Dataset/RUL_FD001.csv"
)

if "Engine_ID" not in rul_df.columns:

    rul_df["Engine_ID"] = np.arange(
        1,
        len(rul_df) + 1
    )

rul_df["RUL"] = rul_df["RUL"].clip(upper=RUL_CAP)


In [18]:
prediction_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": bilstm_pred_30
})

comparison = prediction_df.merge(
    rul_df,
    on="Engine_ID"
)

comparison.head()

,Engine_ID,Predicted_RUL,RUL
0,1,118.561325,112
1,2,114.627914,98
2,3,66.417381,69
3,4,92.165314,82
4,5,103.029358,91


In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)

actual = comparison["RUL"].values
predicted = comparison["Predicted_RUL"].values

mae = mean_absolute_error(
    actual,
    predicted
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        predicted
    )
)

r2 = r2_score(
    actual,
    predicted
)

score = nasa_score(
    actual,
    predicted
)

print("10% Missing - BiLSTM")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


10% Missing - BiLSTM
MAE  : 11.843461036682129
RMSE : 15.980535588652694
R²   : 0.8409721851348877
Score: 591.48393866156


In [20]:
def evaluate(name, actual, predicted):
    predicted = np.asarray(predicted).ravel()
    actual = np.asarray(actual).ravel()

    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
        "Score": nasa_score(actual, predicted)
    }

results_30 = []

# Add your existing BiLSTM result
results_30.append(
    evaluate(
        "BiLSTM",
        actual,
        bilstm_pred_30
    )
)


LSTM

In [21]:
from sklearn.preprocessing import StandardScaler

y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(
    y_train_30.reshape(-1, 1)
).ravel()

y_val_scaled = y_scaler.transform(
    y_val_30.reshape(-1, 1)
).ravel()

In [22]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(42)

lstm_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

lstm_30.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

In [23]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history_lstm_30 = lstm_30.fit(
    X_train_30,
    y_train_scaled,

    validation_data=(
        X_val_30,
        y_val_scaled
    ),

    epochs=100,
    batch_size=64,
    shuffle=False,

    callbacks=[
        early_stop,
        reduce_lr
    ],

    verbose=1
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 34ms/step - loss: 0.8336 - mae: 0.7883 - val_loss: 0.4092 - val_mae: 0.5380 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.4514 - mae: 0.5506 - val_loss: 0.3137 - val_mae: 0.4668 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.4009 - mae: 0.5159 - val_loss: 0.2727 - val_mae: 0.4402 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.3630 - mae: 0.4879 - val_loss: 0.2745 - val_mae: 0.4442 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 0.3438 - mae: 0.4728 - val_loss: 0.2484 - val_mae: 0.4176 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 0.3342 - mae: 0.4646 - val_loss: 0.2426 - val_mae: 0.4124 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 0.3112 - mae: 0.4478 - val_loss: 0.2210 - val_mae: 0.3910 - learning_rate

In [24]:
lstm_pred_scaled = lstm_30.predict(
    X_test_30,
    verbose=0
)

lstm_pred_30 = y_scaler.inverse_transform(
    lstm_pred_scaled
).ravel()

In [25]:
print(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

results_30.append(
    evaluate(
        "LSTM",
        actual,
        lstm_pred_30
    )
)

{'Model': 'LSTM', 'MAE': 11.368947982788086, 'RMSE': np.float64(15.310727635788728), 'R2': 0.854023814201355, 'Score': np.float64(395.0830017593479)}


GRU

In [26]:
from tensorflow.keras.layers import GRU

tf.keras.backend.clear_session()
tf.random.set_seed(42)

gru_30 = Sequential([
    Input(
        shape=(WINDOW, len(MODEL_FEATURES))
    ),

    GRU(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    GRU(32),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

gru_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

gru_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)

gru_pred_30 = gru_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "GRU",
        actual,
        gru_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - loss: 5344.2251 - mae: 62.4069 - val_loss: 2974.9934 - val_mae: 47.2359
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 2068.7268 - mae: 40.0919 - val_loss: 1761.7565 - val_mae: 37.5733
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1791.8470 - mae: 37.4771 - val_loss: 1749.4131 - val_mae: 37.2499
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1802.4242 - mae: 37.5097 - val_loss: 1749.2195 - val_mae: 37.2380
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 1800.8340 - mae: 37.5026 - val_loss: 1749.0277 - val_mae: 37.2245
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1786.3278 - mae: 37.3634 - val_loss: 1749.1188 - val_mae: 37.2317
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1804.9084 - mae: 37.5080 - val_loss: 1749.1378 - val_mae: 37.2338
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1804.4420 - mae: 37.5618 - val_loss: 1748

TCN

In [27]:
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Dropout,
    Add,
    Activation,
    Lambda,
    Dense
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

def tcn_block(
    x,
    filters,
    kernel_size,
    dilation_rate,
    dropout=0.1
):
    residual = x

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    x = Conv1D(
        filters,
        kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)

    x = Dropout(dropout)(x)

    if residual.shape[-1] != filters:
        residual = Conv1D(
            filters,
            1,
            padding="same"
        )(residual)

    x = Add()([x, residual])

    return Activation("relu")(x)

In [28]:
tf.keras.backend.clear_session()
tf.random.set_seed(42)

inputs = Input(
    shape=(WINDOW, len(MODEL_FEATURES))
)

x = tcn_block(inputs, 32, 2, 1)
x = tcn_block(x, 32, 2, 2)
x = tcn_block(x, 32, 2, 4)
x = tcn_block(x, 32, 2, 8)

x = Lambda(
    lambda z: z[:, -1, :]
)(x)

x = Dense(
    32,
    activation="relu"
)(x)

x = Dropout(0.1)(x)

outputs = Dense(1)(x)

tcn_30 = Model(
    inputs,
    outputs
)

tcn_30.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
    metrics=["mae"]
)

tcn_30.fit(
    X_train_30,
    y_train_30,
    validation_data=(X_val_30, y_val_30),
    epochs=100,
    batch_size=64,
    shuffle=True,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ],
    verbose=1
)

tcn_pred_30 = tcn_30.predict(
    X_test_30,
    verbose=0
).ravel()

results_30.append(
    evaluate(
        "TCN",
        actual,
        tcn_pred_30
    )
)

Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 1953.5042 - mae: 34.9301 - val_loss: 486.2952 - val_mae: 17.8803
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 691.9157 - mae: 20.9603 - val_loss: 461.6159 - val_mae: 17.3502
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 625.9332 - mae: 19.8012 - val_loss: 489.3907 - val_mae: 18.3133
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 572.3344 - mae: 18.8294 - val_loss: 467.4723 - val_mae: 18.0116
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - loss: 548.4830 - mae: 18.2978 - val_loss: 483.7322 - val_mae: 18.2445
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 525.1423 - mae: 17.9279 - val_loss: 415.0628 - val_mae: 16.4907
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 494.3354 - mae: 17.2854 - val_loss: 436.2140 - val_mae: 17.1184
Epoch 8/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 479.7450 - mae: 16.8900 - val_loss: 440.9755 - val_mae

Random Forest

In [29]:
X_train_flat_30 = X_train_30.reshape(
    X_train_30.shape[0],
    -1
)

X_test_flat_30 = X_test_30.reshape(
    X_test_30.shape[0],
    -1
)

In [30]:
from sklearn.ensemble import RandomForestRegressor

rf_30 = RandomForestRegressor(
    n_estimators=500,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_30.fit(
    X_train_flat_30,
    y_train_30
)

rf_pred_30 = rf_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "Random Forest",
        actual,
        rf_pred_30
    )
)

XGBoost

In [31]:
from xgboost import XGBRegressor

X_val_flat_30 = X_val_30.reshape(
    X_val_30.shape[0],
    -1
)

xgb_30 = XGBRegressor(
    n_estimators=700,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_30.fit(
    X_train_flat_30,
    y_train_30,
    eval_set=[
        (
            X_val_flat_30,
            y_val_30
        )
    ],
    verbose=False
)

xgb_pred_30 = xgb_30.predict(
    X_test_flat_30
)

results_30.append(
    evaluate(
        "XGBoost",
        actual,
        xgb_pred_30
    )
)

In [32]:
results_30_df = pd.DataFrame(
    results_30
).sort_values("RMSE")

print(results_30_df)

           Model        MAE       RMSE        R2       Score
5        XGBoost  10.469161  13.804777  0.881328  325.440727
2            GRU  10.699149  14.609574  0.867088  407.075489
1           LSTM  11.368948  15.310728  0.854024  395.083002
0         BiLSTM  11.843461  15.980536  0.840972  591.483939
4  Random Forest  13.401240  17.940308  0.799576  852.046250
3            TCN  15.473730  19.752989  0.757028  725.793825
